In [9]:
# new import
import pandera.pandas as pa
from pandera import Column, DataFrameSchema, Check
from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path


API_URL1 = "https://restcountries.com/v3.1/all?fields=cca3,cca2,area,cioc,independent,fifa,landlocked,ccn3,population,region"
API_URL2 = "https://restcountries.com/v3.1/all?fields=cca3,status,subregion,unMember"


# --- Schema ---
countries_schema = pa.DataFrameSchema({
    "cca3": Column(str, [
        Check.str_length(3, 3)  # debe tener 3 letras
    ], nullable=False),
    "cca2": Column(str, [
        Check.str_length(2, 2)  # 2 letras
    ], nullable=True),
    "area": Column(float, [
        Check.ge(0)
    ], nullable=True),
    "cioc": Column(str, nullable=True),
    "independent": Column(bool, nullable=True),
    "fifa": Column(str, nullable=True),
    "landlocked": Column(bool, nullable=True),
    "ccn3": Column(str, [
        Check.str_length(3, 3)  # siempre 3 caracteres
    ], nullable=True),  # acepta None si no existe
    "population": Column(int, [
        Check.ge(0)
    ], nullable=True),
    "region": Column(str, nullable=True),
    "status": Column(str, nullable=True),
    "subregion": Column(str, nullable=True),
    "unMember": Column(bool, nullable=True),
})


@task
def extract_api1():
    response = requests.get(API_URL1)
    response.raise_for_status()
    data = response.json()
    print(f"Se cargaron {len(data)} países (API1)")
    return data


@task
def extract_api2():
    response = requests.get(API_URL2)
    response.raise_for_status()
    data = response.json()
    print(f"Se cargaron {len(data)} países (API2)")
    return data


@task
def transform_and_merge(data, data2):
    df = pd.DataFrame(data)
    df2 = pd.DataFrame(data2)
    df_inner = pd.merge(df, df2, on="cca3", how="inner")

    #  Normalizar ccn3: string de 3 dígitos o None
    if "ccn3" in df_inner.columns:
        df_inner["ccn3"] = (
            df_inner["ccn3"]
            .astype("string")                 # asegurar tipo string
            .str.zfill(3)                     # rellenar con ceros
            .where(df_inner["ccn3"].notna(), None)  # si NaN -> None
        )

    return df_inner


@task
def validate_countries(df: pd.DataFrame) -> pd.DataFrame:
    return countries_schema.validate(df)


@task
def save_csv(df: pd.DataFrame):
    file_path = Path("../stage/countries.csv")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(file_path, index=False)
    print(f"Archivo guardado en {file_path}")


@flow(name="etl_countries_flow")
def etl_countries_flow():
    data1 = extract_api1()
    data2 = extract_api2()
    df = transform_and_merge(data1, data2)
    df = validate_countries(df)  # validación
    save_csv(df)


if __name__ == "__main__":
    etl_countries_flow()


03:41:37.607 | INFO    | Flow run 'debonair-sidewinder' - Beginning flow run 'debonair-sidewinder' for flow 'etl_countries_flow'

Se cargaron 250 países (API1)


03:41:38.764 | INFO    | Task run 'extract_api1-f2d' - Finished in state Completed()

Se cargaron 250 países (API2)


03:41:39.864 | INFO    | Task run 'extract_api2-113' - Finished in state Completed()

03:41:40.127 | INFO    | Task run 'transform_and_merge-3f7' - Finished in state Completed()

03:41:40.381 | INFO    | Task run 'validate_countries-27b' - Finished in state Completed()

Archivo guardado en ..\stage\countries.csv


03:41:40.616 | INFO    | Task run 'save_csv-5e0' - Finished in state Completed()

03:41:40.648 | INFO    | Flow run 'debonair-sidewinder' - Finished in state Completed()